# LeRobot Data Profiler (PySpark)

Profile any LeRobotDataset v3 folder to see whether it is sound enough to train on, and where it deviates from what its metadata claims. <br>
Built against PushT, then run on aloha_sim and an SO-101 dataset to compare sim and real.

## Setup

Datasets: `lerobot/pusht` and `lerobot/aloha_sim_insertion_human` (both simulations).

Run top to bottom. Nothing to edit: each dataset has its own config cell.

## Part 2 · The nine checks

| | Check | Question |
|---|---|---|
| A | Overview | What does the dataset say about itself? |
| B | Episode lengths | Do the sources agree? Any unusual episodes? |
| C | Frame timing | Do timestamps match the declared fps? |
| D | Video windows | Is each video clip as long as its data? |
| E | Action | Right size, sane ranges, no bad values? |
| F | Idle and dead | Did anything move? |
| G | State | Same questions as E, for the robot's readings |
| H | Cameras | How many, how big, do all sources agree? |
| I | Outcomes | Is there a built-in quality signal? |

## Part 3 · Setup (terminal, Claude Code)

```
Clone https://github.com/snudurupati/physical-ai-data-architect into ~/Projects/paida-wk02.
Check out the tag week-02.
Run uv sync to build the environment from the lockfile.
Check that Java 17 or newer is installed (java -version) and print the path JAVA_HOME points to. If Java is missing, do not install anything, just tell me.
Load lerobot/pusht and then lerobot/aloha_sim_insertion_human with LeRobotDataset (from lerobot.datasets.lerobot_dataset import LeRobotDataset) so both download. Print num_episodes and num_frames for each.
Do not change any files in the repo.
```

Expected: `206 25650` and `50 25000`.

## Part 4 · Profile PushT

In [ ]:
import os, glob
from pyspark.sql import SparkSession

spark = (SparkSession
            .builder
                .master("local[*]")
                .appName("paida")
                .config("spark.driver.memory", "16g")
                .getOrCreate())

### Config
The only line to change per dataset is `REPO_ID`.

In [ ]:
import os

REPO_ID = "lerobot/pusht"               # the only line to change per dataset
DATASET_NAME = REPO_ID.split("/")[1]
ROOT = os.path.expanduser(f"~/.cache/huggingface/lerobot/{REPO_ID}")


def files(sub):
    return glob.glob(f"{ROOT}/{sub}")

data_files = files("data/*/*.parquet")
episode_files = files("meta/episodes/*/*.parquet")
info_file = f"{ROOT}/meta/info.json"
video_files = files("videos/*/*/*.mp4")

print(info_file)

### A. Overview
What does the dataset say about itself?

In [ ]:
meta_df = (spark
            .read
            .option("multiline", "true")
            .json(info_file))
meta_df.printSchema()

In [ ]:
meta_df.selectExpr("total_episodes", "total_frames", "total_tasks", "fps", 
                    "features.`observation.state`.names AS state_names",
                    "features.`observation.state`.shape as state_shape",
                    "features.action.names as action_names",
                    "features.action.shape as action_shape"
                    ).show()

### B. Episode lengths
Do the sources agree? Any unusual episodes?

`episode_df`: one row per episode (metadata). `data_df`: one row per frame.

In [ ]:
episode_df = spark.read.parquet(*episode_files)
data_df = spark.read.parquet(*data_files)
episode_df.printSchema()
data_df.printSchema()

In [ ]:
#episode stats from metadata
episode_df.selectExpr("count(episode_index) as total_episodes",
                        "sum(length) as total_frames",
                        "min(length) as min_ep_length",
                        "max(length) as max_ep_length",
                        "avg(length) as avg_ep_length"
                        ).show()

#episode stats from data
data_df.selectExpr("count(distinct episode_index) as total_episodes",
                    "count(frame_index) as total_frames",
                    ).show()

Totals can match while individual episodes don't. Subtract per-episode counts to check.

In [ ]:
# Verify frames per episode between the metadata and data
import pyspark.sql.functions as F

(episode_df.groupBy("episode_index").agg(F.sum("length").alias("frame_count"))
.exceptAll(
data_df.groupBy("episode_index").agg(F.count("frame_index").alias("frame_count"))
)).count()

**Sorted lengths.** Flat means similar episodes. A bend at the ends means unusual ones.

In [ ]:
# one-time: uv add --dev matplotlib
import matplotlib.pyplot as plt

lengths = [r["length"] for r in episode_df.select("length").orderBy("length").collect()]

plt.plot(lengths)
plt.xlabel("episode (sorted by length)")
plt.ylabel("length (frames)")
plt.show()

**Why quantiles?** An average gets dragged by extremes. Percentiles show the typical episode and where the unusual ones start.<br> Example: if the 25th percentile is 99 frames, then 25% of the episodes are shorter than 99 frames.

In [ ]:
import pyspark.sql.functions as F
import matplotlib.pyplot as plt

pcts = [i / 100 for i in range(0, 101)]          # 0.00, 0.01, ... 1.00
vals = (episode_df
        .select(F.percentile_approx("length", pcts, 10000).alias("v"))
        .first()["v"])   
plt.plot([p * 100 for p in pcts], vals)
for q in (25, 50, 75):
    plt.axvline(q, linestyle="--", linewidth=0.8)
    print(f"{q}th percentile: {vals[q]} frames")
plt.xlabel("percentile")
plt.ylabel("length (frames)")
plt.show()

**IQR (interquartile range)** = 75th percentile minus 25th: the width of the middle half of the episodes. Extremes can't stretch it, so it works as a robust ruler for "unusual".

In [ ]:
q1 = vals[25]
q3 = vals[75]
iqr = q3 - q1
print(q1, q3, iqr)

**Fences.** Flag anything more than 1.5 x IQR outside the middle band. The 1.5 is a convention, not a law. Flagged does not mean bad: open it and look.

In [ ]:
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
print(lower, upper)

outliers = (episode_df
            .filter((F.col("length") < lower) | (F.col("length") > upper))
            .select("episode_index", "length")
            .orderBy("length"))
outliers.show()

In [ ]:
# **Look at the flagged episodes** (terminal):

# uv run lerobot-dataset-viz --repo-id lerobot/pusht --episode-index 184
# uv run lerobot-dataset-viz --repo-id lerobot/pusht --episode-index 174

### C. Frame timing
`timestamp * fps` should equal `frame_index`.

Source: [`dataset_writer.py`, `add_frame`](https://github.com/huggingface/lerobot/blob/main/src/lerobot/datasets/dataset_writer.py)

In [ ]:
fps, total_frames = meta_df.select("fps", "total_frames").first()
data_df2 = data_df.withColumn("calc_fr_idx",
                    F.round(F.col("timestamp")*F.lit(fps))
                    .cast("int"))
frame_count = data_df2.where("calc_fr_idx == frame_index").count()
assert frame_count == total_frames

### D. Video windows
Is each episode's video clip as long as its data?

Source: the clip length is measured from the encoded file (`get_video_duration_in_s`), in the same file.

In [ ]:
from pyspark.sql import functions as F

fps = meta_df.first()["fps"]
cams = sorted({c[len("videos/"):-len("/from_timestamp")]
               for c in episode_df.schema.fieldNames()
               if c.startswith("videos/") and c.endswith("/from_timestamp")})

# 1. frames per episode, counted from the data
frames_df = data_df.groupBy("episode_index").agg(F.count("*").alias("n_frames"))

# 2. one column per camera: video clip length in seconds
clip_cols = [(F.col(f"`videos/{c}/to_timestamp`") - F.col(f"`videos/{c}/from_timestamp`")).alias(f"clip_{c}")
             for c in cams]

clips_df = episode_df.select("episode_index", *clip_cols)

# 3. join, then compare each camera to frames / fps
df = clips_df.join(frames_df, "episode_index")
assert df.count() == frames_df.count()

# Tolerance instead of == 0: float rounding leaves tiny gaps, so allow half a frame (0.05 s at 10 fps).
# abs() because the clip can be shorter or longer than the data. Flag if the gap exceeds the tolerance.
tol = 0.5 / fps   # half a frame, in seconds
bad = [F.abs(F.col(f"`clip_{c}`") - F.col("n_frames") / fps) > tol for c in cams]
flagged_df = df.where(F.greatest(*bad)) if len(bad) > 1 else df.where(bad[0])
flagged_df.show()

### E. Action
Right size? Sane ranges? Nulls or NaNs?

One helper for action and state: both are a list of numbers per frame.

In [ ]:
def vec_profile(df, name):
    """Profile an array column: lengths, nulls, NaNs, per-dimension min/max."""
    c = F.col(f"`{name}`")                    # backticks: the name may contain a dot
    n = F.size(c)                             # array length per row
    summary_df = df.agg(
        F.count("*").alias("rows"),
        F.sum(c.isNull().cast("int")).alias("null_rows"),
        F.min(n).alias("min_len"),
        F.max(n).alias("max_len"),
        F.sum((n == 0).cast("int")).alias("empty_rows"),
    )
    per_dim_df = (df.select(F.posexplode(c).alias("dim", "value"))   # one row per (frame, position)
        .groupBy("dim")
        .agg(F.min("value").alias("min"),
             F.max("value").alias("max"),
             F.sum(F.isnan("value").cast("int")).alias("nan_values"),
             F.sum(F.col("value").isNull().cast("int")).alias("null_values"))
        .orderBy("dim"))
    return summary_df, per_dim_df

def declared_len(name):
    """Length declared in info.json: features.<name>.shape[0]."""
    return meta_df.select(F.col(f"features.`{name}`.shape")[0]).first()[0]

In [ ]:
action_sum_df, action_dim_df = vec_profile(data_df, "action")
action_sum_df.show()
action_dim_df.show()

s = action_sum_df.first()
dec = declared_len("action")
bad_values = action_dim_df.agg(F.sum("nan_values") + F.sum("null_values")).first()[0]
ok = (s["min_len"] == s["max_len"] == dec) and s["empty_rows"] == 0 and s["null_rows"] == 0 and bad_values == 0
print("E action:", "ok" if ok else "FLAG", f"(declared length {dec})")

### F. Idle and dead
Did anything move?

Range = max minus min of a joint within an episode. **Idle episode:** even its busiest joint barely moved. **Dead joint:** barely moved in any episode. `EPS` is my arbitrary cutoff, in the data's own units.

In [ ]:
EPS = 1e-6   # "about zero": a range below this means no movement

# one row per (frame, joint position), then range per episode and joint
range_df = (data_df.select("episode_index", F.posexplode("action").alias("joint", "v"))
    .groupBy("episode_index", "joint")
    .agg((F.max("v") - F.min("v")).alias("range")))

print("smallest per-episode, per-joint range:", range_df.agg(F.min("range")).first()[0])

# idle episode: even its most-moving joint barely moved
idle_df = (range_df.groupBy("episode_index").agg(F.max("range").alias("max_range"))
           .where(F.col("max_range") < EPS))

# dead joint: it barely moved in any episode
dead_df = (range_df.groupBy("joint").agg(F.max("range").alias("max_range"))
           .where(F.col("max_range") < EPS))

n_idle, n_dead = idle_df.count(), dead_df.count()
print("idle episodes:", n_idle, "| dead joints:", n_dead)
print("F idle/dead:", "ok" if n_idle == 0 and n_dead == 0 else "FLAG")

### G. State
Same questions as E, for the robot's own readings.

In [ ]:
state_sum_df, state_dim_df = vec_profile(data_df, "observation.state")
state_sum_df.show()
state_dim_df.show()

ss = state_sum_df.first()
dec_s = declared_len("observation.state")
bad_s = state_dim_df.agg(F.sum("nan_values") + F.sum("null_values")).first()[0]
ok_s = (ss["min_len"] == ss["max_len"] == dec_s) and ss["empty_rows"] == 0 and ss["null_rows"] == 0 and bad_s == 0
print("G state:", "ok" if ok_s else "FLAG", f"(declared length {dec_s})")

dec = declared_len("action")     # also defined in E; repeated so this cell runs alone
print("state size == action size:", dec_s == dec)

### H. Cameras
How many, how big, and do all three sources agree?

In [ ]:
import os
from collections import Counter

# 1. cameras declared in info.json: features whose dtype is "video"
features = meta_df.first()["features"].asDict(recursive=True)
video_feats = {k: v for k, v in features.items() if v["dtype"] == "video"}
for k, v in video_feats.items():
    print(k, "| shape:", v["shape"], "| info:", v.get("video_info"))

# 2. the same cameras seen from the other two sources
cams_meta = sorted(video_feats)                                   # info.json
cams_cols = sorted(cams)                                          # episode columns (from check D)
cams_disk = sorted({os.path.relpath(p, ROOT).split(os.sep)[1] for p in video_files})   # folders on disk

# 3. mp4 files per camera
mp4_per_cam = Counter(os.path.relpath(p, ROOT).split(os.sep)[1] for p in video_files)
print("mp4 files per camera:", dict(mp4_per_cam))

print("H cameras:", "ok" if cams_meta == cams_cols == cams_disk else "FLAG")
print("  info.json:", cams_meta, "| episode columns:", cams_cols, "| on disk:", cams_disk)

### I. Outcomes
Does the dataset carry its own quality signal?

In [ ]:
outcome_cols = [c for c in ("next.reward", "next.done", "next.success") if c in data_df.columns]
print("present:", outcome_cols)

n_eps = episode_df.count()
for name in outcome_cols:
    c = F.col(f"`{name}`")                                   # backticks: dotted name

    # aggregate per episode first
    per_ep_df = data_df.groupBy("episode_index").agg(
        F.countDistinct(c).alias("n_distinct"),              # distinct values inside the episode
        F.max(c.cast("double")).alias("mx"))                 # highest value inside the episode

    r = per_ep_df.agg(
        F.sum((F.col("n_distinct") > 1).cast("int")).alias("varies_within"),
        F.sum((F.col("mx") > 0).cast("int")).alias("any_positive")).first()

    values = [row[0] for row in data_df.select(c).distinct().limit(10).collect()]
    print(f"{name}: values {values} | varies within {r['varies_within']}/{n_eps} episodes"
          f" | any positive in {r['any_positive']}/{n_eps} episodes")
    

### Save this run's summary

In [ ]:
import json, pathlib

timing_mismatches = data_df.where(
    F.round(F.col("timestamp") * fps).cast("int") != F.col("frame_index")).count()   # check C
success_eps = (data_df.groupBy("episode_index")
               .agg(F.max(F.col("`next.success`").cast("int")).alias("s"))
               .where("s > 0").count()) if "next.success" in data_df.columns else None

summary = {
    "dataset": DATASET_NAME,
    "fps": fps,
    "episodes": episode_df.count(),
    "frames": data_df.count(),
    "action_size": dec,                       # from E
    "state_size": dec_s,                      # from G
    "cameras": ", ".join(cams),
    "video_codec": next(iter(video_feats.values())).get("video_info", {}).get("video.codec"),
    "timing_mismatches": timing_mismatches,   # C
    "video_window_mismatches": flagged_df.count(),   # D
    "idle_episodes": n_idle,                  # F
    "dead_joints": n_dead,                    # F
    "success_episodes": success_eps,          # I
}
out = pathlib.Path("results"); out.mkdir(exist_ok=True)
(out / f"{DATASET_NAME}.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

## Part 5 · Hunch: reward measures demo quality

Better demos should score higher. Long demos should score lower.

In [ ]:
HAS_REWARD = "next.reward" in data_df.columns   # not every dataset has a reward column

if HAS_REWARD:
    r = F.col("`next.reward`")

    reward_df = (data_df.groupBy("episode_index").agg(
        F.max(r).alias("max_reward"),
        F.mean(r).alias("mean_reward"),
        F.max_by(r, "frame_index").alias("final_reward"),   # reward at the last frame
        F.count("*").alias("n_frames")))

    # spread across the episodes
    reward_df.select("max_reward", "mean_reward", "final_reward").summary(
        "min", "25%", "50%", "75%", "max").show()

    # worst and best five by final reward (candidates to open in Rerun)
    reward_df.orderBy("final_reward").show(5)
    reward_df.orderBy(F.col("final_reward").desc()).show(5)
else:
    print("skipped: no next.reward column in this dataset")

**Two plots.**

- **Left:** each dot is one episode. Longer episode (x) vs its mean reward (y). If long demos scored lower, the dots would slope down.
- **Right (bar chart):** a histogram. x is the final reward, y is how many episodes ended in that range. A tall bar means many episodes ended with about that reward.

In [ ]:
if HAS_REWARD:
    import matplotlib.pyplot as plt

    pdf = reward_df.toPandas()          
    corr = pdf["n_frames"].corr(pdf["mean_reward"])

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))

    # left: is mean reward related to episode length?
    ax[0].scatter(pdf["n_frames"], pdf["mean_reward"], s=14, alpha=0.7)
    for ep in (174, 184):                                    # PushT length outliers, if present
        if (pdf["episode_index"] == ep).any():
            row = pdf[pdf["episode_index"] == ep].iloc[0]
            ax[0].annotate(str(ep), (row["n_frames"], row["mean_reward"]))
    ax[0].set_xlabel("episode length (frames)")
    ax[0].set_ylabel("mean reward")
    ax[0].set_title(f"length vs mean reward (corr {corr:.2f})")

    # right: how different are the final rewards?
    ax[1].hist(pdf["final_reward"], bins=30)
    ax[1].set_xlabel("final reward")
    ax[1].set_ylabel("episodes")
    ax[1].set_title("final reward per episode")

    plt.tight_layout()
    plt.show()
else:
    print("skipped: no next.reward column in this dataset")

In [ ]:
# **Look at the extremes** (terminal):

# uv run lerobot-dataset-viz --repo-id lerobot/pusht --episode-index 95
# uv run lerobot-dataset-viz --repo-id lerobot/pusht --episode-index 32
# uv run lerobot-dataset-viz --repo-id lerobot/pusht --episode-index 124

### Second try: does effort, measured as action path length, explain reward?

In [ ]:
if HAS_REWARD and dec == 2:                      # path length here is written for 2 action values
    from pyspark.sql.window import Window
    import matplotlib.pyplot as plt

    w = Window.partitionBy("episode_index").orderBy("frame_index")
    a = F.col("action")

    # distance between this action and the previous one, per frame
    step = F.sqrt((a[0] - F.lag(a[0]).over(w))**2 + (a[1] - F.lag(a[1]).over(w))**2)

    path_df = (data_df.withColumn("step", step)
               .groupBy("episode_index")
               .agg(F.sum("step").alias("path_len")))          # first frame has a null step, which sum ignores

    effort_df = reward_df.join(path_df, "episode_index")
    pdf2 = effort_df.toPandas()

    c_final = pdf2["path_len"].corr(pdf2["final_reward"])
    c_mean = pdf2["path_len"].corr(pdf2["mean_reward"])

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for axis, col, c in ((ax[0], "final_reward", c_final), (ax[1], "mean_reward", c_mean)):
        axis.scatter(pdf2["path_len"], pdf2[col], s=14, alpha=0.7)
        for ep in (95, 32, 124):                                # PushT episodes inspected, if present
            if (pdf2["episode_index"] == ep).any():
                r = pdf2[pdf2["episode_index"] == ep].iloc[0]
                axis.annotate(str(ep), (r["path_len"], r[col]))
        axis.set_xlabel("action path length")
        axis.set_ylabel(col)
        axis.set_title(f"path length vs {col} (corr {c:.2f})")
    plt.tight_layout()
    plt.show()
else:
    print("skipped: needs next.reward and a 2-value action")

### Verdict on the reward hunch

**Hunch:** reward measures demo quality. **Answer: no.**

- No link to length (corr about -0.01) or to action path length (about -0.02 and 0.05).
- Every episode ends in a narrow band (0.81 to 0.95), so it barely separates demos.
- The extremes I opened (95, 32, 124) matched the final fit of the T, not how tidy the demo looked.

Reward looks like a score for how well the T fits the outline at the end. Not a quality label. Caveat: three episodes watched, not a proof.

## Part 6 · Same checks, new dataset

Only the config cell changes. Run every cell below.

In [ ]:
import os

REPO_ID = "lerobot/aloha_sim_insertion_human"               # the only line to change per dataset
DATASET_NAME = REPO_ID.split("/")[1]
ROOT = os.path.expanduser(f"~/.cache/huggingface/lerobot/{REPO_ID}")


def files(sub):
    return glob.glob(f"{ROOT}/{sub}")

data_files = files("data/*/*.parquet")
episode_files = files("meta/episodes/*/*.parquet")
info_file = f"{ROOT}/meta/info.json"
video_files = files("videos/*/*/*.mp4")

print(info_file)

### A. Overview
What does the dataset say about itself?

In [ ]:
meta_df = (spark
            .read
            .option("multiline", "true")
            .json(info_file))
meta_df.printSchema()

In [ ]:
meta_df.selectExpr("total_episodes", "total_frames", "total_tasks", "fps", 
                    "features.`observation.state`.names AS state_names",
                    "features.`observation.state`.shape as state_shape",
                    "features.action.names as action_names",
                    "features.action.shape as action_shape"
                    ).show()

### B. Episode lengths
Do the sources agree? Any unusual episodes?

`episode_df`: one row per episode (metadata). `data_df`: one row per frame.

In [ ]:
episode_df = spark.read.parquet(*episode_files)
data_df = spark.read.parquet(*data_files)
episode_df.printSchema()
data_df.printSchema()

In [ ]:
#episode stats from metadata
episode_df.selectExpr("count(episode_index) as total_episodes",
                        "sum(length) as total_frames",
                        "min(length) as min_ep_length",
                        "max(length) as max_ep_length",
                        "avg(length) as avg_ep_length"
                        ).show()

#episode stats from data
data_df.selectExpr("count(distinct episode_index) as total_episodes",
                    "count(frame_index) as total_frames",
                    ).show()

Totals can match while individual episodes don't. Subtract per-episode counts to check.

In [ ]:
# Verify frames per episode between the metadata and data
import pyspark.sql.functions as F

(episode_df.groupBy("episode_index").agg(F.sum("length").alias("frame_count"))
.exceptAll(
data_df.groupBy("episode_index").agg(F.count("frame_index").alias("frame_count"))
)).count()

**Sorted lengths.** Flat means similar episodes. A bend at the ends means unusual ones.

In [ ]:
# one-time: uv add --dev matplotlib
import matplotlib.pyplot as plt

lengths = [r["length"] for r in episode_df.select("length").orderBy("length").collect()]

plt.plot(lengths)
plt.xlabel("episode (sorted by length)")
plt.ylabel("length (frames)")
plt.show()

**Why quantiles?** An average gets dragged by extremes. Percentiles show the typical episode and where the unusual ones start. Example: if the 25th percentile is 99 frames, then 25% of the episodes are shorter than 99 frames.

In [ ]:
import pyspark.sql.functions as F
import matplotlib.pyplot as plt

pcts = [i / 100 for i in range(0, 101)]          # 0.00, 0.01, ... 1.00
vals = (episode_df
        .select(F.percentile_approx("length", pcts, 10000).alias("v"))
        .first()["v"])   
plt.plot([p * 100 for p in pcts], vals)
for q in (25, 50, 75):
    plt.axvline(q, linestyle="--", linewidth=0.8)
    print(f"{q}th percentile: {vals[q]} frames")
plt.xlabel("percentile")
plt.ylabel("length (frames)")
plt.show()

**IQR (interquartile range)** = 75th percentile minus 25th: the width of the middle half of the episodes. Extremes can't stretch it, so it works as a robust ruler for "unusual".

In [ ]:
q1 = vals[25]
q3 = vals[75]
iqr = q3 - q1
print(q1, q3, iqr)

**Fences.** Flag anything more than 1.5 x IQR outside the middle band. The 1.5 is a convention, not a law. Flagged does not mean bad: open it and look.

In [ ]:
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
print(lower, upper)

outliers = (episode_df
            .filter((F.col("length") < lower) | (F.col("length") > upper))
            .select("episode_index", "length")
            .orderBy("length"))
outliers.show()

### C. Frame timing
`timestamp * fps` should equal `frame_index`.

Source: [`dataset_writer.py`, `add_frame`](https://github.com/huggingface/lerobot/blob/main/src/lerobot/datasets/dataset_writer.py)

In [ ]:
fps, total_frames = meta_df.select("fps", "total_frames").first()
data_df2 = data_df.withColumn("calc_fr_idx",
                    F.round(F.col("timestamp")*F.lit(fps))
                    .cast("int"))
frame_count = data_df2.where("calc_fr_idx == frame_index").count()
assert frame_count == total_frames

### D. Video windows
Is each episode's video clip as long as its data?

Source: the clip length is measured from the encoded file (`get_video_duration_in_s`), in the same file.

In [ ]:
from pyspark.sql import functions as F

fps = meta_df.first()["fps"]
cams = sorted({c[len("videos/"):-len("/from_timestamp")]
               for c in episode_df.schema.fieldNames()
               if c.startswith("videos/") and c.endswith("/from_timestamp")})

# 1. frames per episode, counted from the data
frames_df = data_df.groupBy("episode_index").agg(F.count("*").alias("n_frames"))

# 2. one column per camera: video clip length in seconds
clip_cols = [(F.col(f"`videos/{c}/to_timestamp`") - F.col(f"`videos/{c}/from_timestamp`")).alias(f"clip_{c}")
             for c in cams]

clips_df = episode_df.select("episode_index", *clip_cols)

# 3. join, then compare each camera to frames / fps
df = clips_df.join(frames_df, "episode_index")
assert df.count() == frames_df.count()

# Tolerance instead of == 0: float rounding leaves tiny gaps, so allow half a frame (0.05 s at 10 fps).
# abs() because the clip can be shorter or longer than the data. Flag if the gap exceeds the tolerance.
tol = 0.5 / fps   # half a frame, in seconds
bad = [F.abs(F.col(f"`clip_{c}`") - F.col("n_frames") / fps) > tol for c in cams]
flagged_df = df.where(F.greatest(*bad)) if len(bad) > 1 else df.where(bad[0])
flagged_df.show()

### E. Action
Right size? Sane ranges? Nulls or NaNs?

One helper for action and state: both are a list of numbers per frame.

In [ ]:
def vec_profile(df, name):
    """Profile an array column: lengths, nulls, NaNs, per-dimension min/max."""
    c = F.col(f"`{name}`")                    # backticks: the name may contain a dot
    n = F.size(c)                             # array length per row
    summary_df = df.agg(
        F.count("*").alias("rows"),
        F.sum(c.isNull().cast("int")).alias("null_rows"),
        F.min(n).alias("min_len"),
        F.max(n).alias("max_len"),
        F.sum((n == 0).cast("int")).alias("empty_rows"),
    )
    per_dim_df = (df.select(F.posexplode(c).alias("dim", "value"))   # one row per (frame, position)
        .groupBy("dim")
        .agg(F.min("value").alias("min"),
             F.max("value").alias("max"),
             F.sum(F.isnan("value").cast("int")).alias("nan_values"),
             F.sum(F.col("value").isNull().cast("int")).alias("null_values"))
        .orderBy("dim"))
    return summary_df, per_dim_df

def declared_len(name):
    """Length declared in info.json: features.<name>.shape[0]."""
    return meta_df.select(F.col(f"features.`{name}`.shape")[0]).first()[0]

In [ ]:
action_sum_df, action_dim_df = vec_profile(data_df, "action")
action_sum_df.show()
action_dim_df.show()

s = action_sum_df.first()
dec = declared_len("action")
bad_values = action_dim_df.agg(F.sum("nan_values") + F.sum("null_values")).first()[0]
ok = (s["min_len"] == s["max_len"] == dec) and s["empty_rows"] == 0 and s["null_rows"] == 0 and bad_values == 0
print("E action:", "ok" if ok else "FLAG", f"(declared length {dec})")

### F. Idle and dead
Did anything move?

Range = max minus min of a joint within an episode. **Idle episode:** even its busiest joint barely moved. **Dead joint:** barely moved in any episode. `EPS` is my arbitrary cutoff, in the data's own units.

In [ ]:
EPS = 1e-6   # "about zero": a range below this means no movement

# one row per (frame, joint position), then range per episode and joint
range_df = (data_df.select("episode_index", F.posexplode("action").alias("joint", "v"))
    .groupBy("episode_index", "joint")
    .agg((F.max("v") - F.min("v")).alias("range")))

print("smallest per-episode, per-joint range:", range_df.agg(F.min("range")).first()[0])

# idle episode: even its most-moving joint barely moved
idle_df = (range_df.groupBy("episode_index").agg(F.max("range").alias("max_range"))
           .where(F.col("max_range") < EPS))

# dead joint: it barely moved in any episode
dead_df = (range_df.groupBy("joint").agg(F.max("range").alias("max_range"))
           .where(F.col("max_range") < EPS))

n_idle, n_dead = idle_df.count(), dead_df.count()
print("idle episodes:", n_idle, "| dead joints:", n_dead)
print("F idle/dead:", "ok" if n_idle == 0 and n_dead == 0 else "FLAG")

### G. State
Same questions as E, for the robot's own readings.

In [ ]:
state_sum_df, state_dim_df = vec_profile(data_df, "observation.state")
state_sum_df.show()
state_dim_df.show()

ss = state_sum_df.first()
dec_s = declared_len("observation.state")
bad_s = state_dim_df.agg(F.sum("nan_values") + F.sum("null_values")).first()[0]
ok_s = (ss["min_len"] == ss["max_len"] == dec_s) and ss["empty_rows"] == 0 and ss["null_rows"] == 0 and bad_s == 0
print("G state:", "ok" if ok_s else "FLAG", f"(declared length {dec_s})")

dec = declared_len("action")     # also defined in E; repeated so this cell runs alone
print("state size == action size:", dec_s == dec)

### H. Cameras
How many, how big, and do all three sources agree?

In [ ]:
import os
from collections import Counter

# 1. cameras declared in info.json: features whose dtype is "video"
features = meta_df.first()["features"].asDict(recursive=True)
video_feats = {k: v for k, v in features.items() if v["dtype"] == "video"}
for k, v in video_feats.items():
    print(k, "| shape:", v["shape"], "| info:", v.get("video_info"))

# 2. the same cameras seen from the other two sources
cams_meta = sorted(video_feats)                                   # info.json
cams_cols = sorted(cams)                                          # episode columns (from check D)
cams_disk = sorted({os.path.relpath(p, ROOT).split(os.sep)[1] for p in video_files})   # folders on disk

# 3. mp4 files per camera
mp4_per_cam = Counter(os.path.relpath(p, ROOT).split(os.sep)[1] for p in video_files)
print("mp4 files per camera:", dict(mp4_per_cam))

print("H cameras:", "ok" if cams_meta == cams_cols == cams_disk else "FLAG")
print("  info.json:", cams_meta, "| episode columns:", cams_cols, "| on disk:", cams_disk)

### I. Outcomes
Does the dataset carry its own quality signal?

In [ ]:
outcome_cols = [c for c in ("next.reward", "next.done", "next.success") if c in data_df.columns]
print("present:", outcome_cols)

n_eps = episode_df.count()
for name in outcome_cols:
    c = F.col(f"`{name}`")                                   # backticks: dotted name

    # aggregate per episode first
    per_ep_df = data_df.groupBy("episode_index").agg(
        F.countDistinct(c).alias("n_distinct"),              # distinct values inside the episode
        F.max(c.cast("double")).alias("mx"))                 # highest value inside the episode

    r = per_ep_df.agg(
        F.sum((F.col("n_distinct") > 1).cast("int")).alias("varies_within"),
        F.sum((F.col("mx") > 0).cast("int")).alias("any_positive")).first()

    values = [row[0] for row in data_df.select(c).distinct().limit(10).collect()]
    print(f"{name}: values {values} | varies within {r['varies_within']}/{n_eps} episodes"
          f" | any positive in {r['any_positive']}/{n_eps} episodes")

Reward cell on a dataset with no reward column:

In [ ]:
HAS_REWARD = "next.reward" in data_df.columns   # not every dataset has a reward column

if HAS_REWARD:
    r = F.col("`next.reward`")

    reward_df = (data_df.groupBy("episode_index").agg(
        F.max(r).alias("max_reward"),
        F.mean(r).alias("mean_reward"),
        F.max_by(r, "frame_index").alias("final_reward"),   # reward at the last frame
        F.count("*").alias("n_frames")))

    # spread across the episodes
    reward_df.select("max_reward", "mean_reward", "final_reward").summary(
        "min", "25%", "50%", "75%", "max").show()

    # worst and best five by final reward (candidates to open in Rerun)
    reward_df.orderBy("final_reward").show(5)
    reward_df.orderBy(F.col("final_reward").desc()).show(5)
else:
    print("skipped: no next.reward column in this dataset")

### Save this run's summary

In [ ]:
import json, pathlib

timing_mismatches = data_df.where(
    F.round(F.col("timestamp") * fps).cast("int") != F.col("frame_index")).count()   # check C
success_eps = (data_df.groupBy("episode_index")
               .agg(F.max(F.col("`next.success`").cast("int")).alias("s"))
               .where("s > 0").count()) if "next.success" in data_df.columns else None

summary = {
    "dataset": DATASET_NAME,
    "fps": fps,
    "episodes": episode_df.count(),
    "frames": data_df.count(),
    "action_size": dec,                       # from E
    "state_size": dec_s,                      # from G
    "cameras": ", ".join(cams),
    "video_codec": next(iter(video_feats.values())).get("video_info", {}).get("video.codec"),
    "timing_mismatches": timing_mismatches,   # C
    "video_window_mismatches": flagged_df.count(),   # D
    "idle_episodes": n_idle,                  # F
    "dead_joints": n_dead,                    # F
    "success_episodes": success_eps,          # I
}
out = pathlib.Path("results"); out.mkdir(exist_ok=True)
(out / f"{DATASET_NAME}.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

**The flat line in B.** Every episode is 500 frames.

Source: [ACT `constants.py`](https://github.com/tonyzhaozh/act/blob/main/constants.py), `sim_insertion_human`: `episode_len` 500, `DT` 0.02.

## Part 7 · Hunch: the end of every episode is idle

Tail motion = sum over joints of (max minus min) of `action`, last 100 frames.
Head motion = same, first 400 frames.

In [ ]:
L = episode_df.agg(F.max("length")).first()[0]      # 500 for this dataset
TAIL = 100                                          # last 100 frames

def motion_df(cond, name):
    """Sum over joints of (max - min) of the action, over the frames chosen by cond."""
    return (data_df.where(cond)
        .select("episode_index", F.posexplode("action").alias("joint", "v"))
        .groupBy("episode_index", "joint")
        .agg((F.max("v") - F.min("v")).alias("r"))
        .groupBy("episode_index")
        .agg(F.sum("r").alias(name)))

tail_df = motion_df(F.col("frame_index") >= L - TAIL, "tail_motion")
head_df = motion_df(F.col("frame_index") <  L - TAIL, "head_motion")
tail_check_df = tail_df.join(head_df, "episode_index")

print("least movement at the end:")
tail_check_df.orderBy("tail_motion").show(3)
print("most movement at the end:")
tail_check_df.orderBy(F.col("tail_motion").desc()).show(3)

In [ ]:
mid = tail_check_df.approxQuantile("tail_motion", [0.5], 0.0)[0]       # median tail motion
(tail_check_df.withColumn("dist", F.abs(F.col("tail_motion") - mid))
    .orderBy("dist").select("episode_index", "tail_motion", "head_motion").show(3))

In [ ]:
# **Look at the episodes** (terminal):

# uv run lerobot-dataset-viz --repo-id lerobot/aloha_sim_insertion_human --episode-index 3
# uv run lerobot-dataset-viz --repo-id lerobot/aloha_sim_insertion_human --episode-index 5
# uv run lerobot-dataset-viz --repo-id lerobot/aloha_sim_insertion_human --episode-index 47
# uv run lerobot-dataset-viz --repo-id lerobot/aloha_sim_insertion_human --episode-index 46
# uv run lerobot-dataset-viz --repo-id lerobot/aloha_sim_insertion_human --episode-index 44

### Hunch: one joint stays steady
The joint that moves the claw forward should swing the same amount in every episode.

In [ ]:
joint_tail_df = (data_df.where(F.col("frame_index") >= L - TAIL)
    .select("episode_index", F.posexplode("action").alias("joint", "v"))
    .groupBy("episode_index", "joint").agg((F.max("v") - F.min("v")).alias("r")))

# per joint: how much does its tail range vary across the episodes?
(joint_tail_df.groupBy("joint")
    .agg(F.min("r").alias("min"), F.percentile_approx("r", 0.5).alias("median"),
         F.max("r").alias("max"), F.stddev("r").alias("std"))
    .orderBy("joint").show(14))

# the sampled episodes side by side, one column per episode
(joint_tail_df.where(F.col("episode_index").isin(5, 47, 46, 3, 44))
    .groupBy("joint").pivot("episode_index").agg(F.first("r"))
    .orderBy("joint").show(14))

**Verdict on the joint hunch.** Hunch: one joint stays steady and the rest cause the variation. **Answer: no.**

- The busiest joints (2, 4, 9, 11) have a median tail swing under 0.02 but a max of 0.2 to 0.3, ten to forty times the median.
- The steady ones (6, 13) are steady because they barely move at all.
- Corrections use different joints: episode 3's big swing is on joint 12, episode 44's on joint 2.

The variation is spread across several joints, in different combinations.

Which joints are these?

In [ ]:
meta_df.selectExpr("posexplode(features.action.names.motors) as (idx, name)").show(14, False)

## Part 8 · Summary

In [ ]:
# Compare results across datasets
import pandas as pd, glob

rows = [json.load(open(p)) for p in sorted(glob.glob("results/*.json"))]
compare_pdf = pd.DataFrame(rows).set_index("dataset").T      # datasets as columns
compare_pdf

## Takeaways

1. Profile before you curate.
2. Measure what varies.
3. Let the data answer the hunch.